# 论文阅读助手

> 给定 doc_id 后分段读取全文，抽取方法、数据、结论、局限

**场景**: 用户已知论文 doc_id，需要分段读取全文并抽取关键信息（方法、数据、结论、局限）。比直接综述 Agent 更基础，也更容易跑通。

**使用接口**: content

**预估调用量**: ~5–12 次 API 调用 / 一篇论文

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
!pip install httpx anthropic
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # 替换为你的真实值


## Step 2: 分段读取全文

循环调用 content 接口，用 next_offset 拼接完整全文


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def read_full_text(doc_id: str, chunk_size: int = 4000) -> str:
    """\u5faa\u73af\u8bfb\u53d6\u5168\u6587\uff0c\u76f4\u5230 more=false"""
    full_text = []
    offset = 0
    async with httpx.AsyncClient(timeout=30) as client:
        while True:
            resp = await client.get(
                f"{BASE}/content", headers=HEADERS,
                params={"doc_id": doc_id, "offset": offset, "limit": chunk_size}
            )
            resp.raise_for_status()
            data = resp.json()
            full_text.append(data["text"])
            if not data.get("more", False):
                break
            offset = data["next_offset"]
    return "".join(full_text)

# \u5148\u901a\u8fc7 agentic-search \u83b7\u53d6\u771f\u5b9e doc_id
hits_resp = await client.post(f"{BASE}/agentic-search", headers=HEADERS, json={"query": "AlphaFold2", "top_k": 1})
hits_resp.raise_for_status()
doc_id = hits_resp.json()["hits"][0]["doc_id"]
text = asyncio.run(read_full_text(doc_id))
print(f"Full text length: {len(text)} chars")
print(f"Preview: {text[:300]}...")


## Step 3: LLM 抽取结构化信息

将全文传给 LLM，抽取方法、数据、结论、局限


In [ ]:
from anthropic import Anthropic

client = Anthropic()

def extract_structure(full_text: str) -> str:
    """\u7528 LLM \u62bd\u53d6\u8bba\u6587\u7ed3\u6784\u5316\u4fe1\u606f"""
    # \u5982\u679c\u5168\u6587\u592a\u957f\uff0c\u53d6\u524d 15000 \u5b57\u7b26
    content = full_text[:15000] if len(full_text) > 15000 else full_text

    msg = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=3000,
        messages=[{
            "role": "user",
            "content": f"""\u8bf7\u9605\u8bfb\u4ee5\u4e0b\u8bba\u6587\u5168\u6587\uff0c\u63d0\u53d6\u4ee5\u4e0b\u56db\u4e2a\u65b9\u9762\u7684\u5173\u952e\u4fe1\u606f\uff1a

1. **\u65b9\u6cd5**: \u6838\u5fc3\u6280\u672f\u65b9\u6cd5\u548c\u521b\u65b0\u70b9
2. **\u6570\u636e**: \u4f7f\u7528\u7684\u6570\u636e\u96c6\u3001\u5b9e\u9a8c\u8bbe\u7f6e\u3001\u5173\u952e\u6570\u503c
3. **\u7ed3\u8bba**: \u4e3b\u8981\u53d1\u73b0\u548c\u8d21\u732e
4. **\u5c40\u9650**: \u5df2\u77e5\u5c40\u9650\u548c\u672a\u6765\u5de5\u4f5c\u65b9\u5411

\u8bba\u6587\u5168\u6587:
{content}"""
        }]
    )
    return msg.content[0].text

report = extract_structure(text)
print(report)


## 注意事项

- content 接口返回 {text, next_offset, more}，循环读取直到 more=false
- 建议 chunk_size=4000，避免单次请求超时
- 如果全文超过 LLM 上下文窗口，可分段抽取后合并
- 这是最基础的论文阅读流程，可作为更复杂 Agent 的子模块
- 部分论文可能无全文（content 返回 404），需做异常处理


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
